# 04_evaluation.ipynb

**Title:** Model Evaluation & Final Test Split Assessment  
**Owner:** Yesid Cardenas Marin (T2)  
**Phase:** Late  
**Reads:**  
- `outputs/predictions/02-lr_val.parquet`  
- `outputs/predictions/03-nn_val.parquet`  
- `artifacts/logreg.joblib`  
- `artifacts/nn_model.keras`  
- `shared.load_features("test", allow_test=True)`  

**Writes:**  
- `outputs/predictions/04-test_predictions.parquet`  
- `outputs/tables/04-eval_metrics_comparison.csv`  
- `outputs/figures/04-eval_roc_comparison.png`  
- `outputs/figures/04-eval_confusion_matrices.png`  
- `outputs/figures/04-eval_comparison_metrics.png`

In [1]:
import sys
import pathlib

# Path bootstrap to import shared utilities from src/
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from keras.models import load_model
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

from shared import PATHS, SEED, compute_metrics, load_features

# Configure global plot styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["font.size"] = 10

# Resolve paths safely using fallback options (prevents KeyError)
base_dir = pathlib.Path.cwd().parent

outputs_dir = PATHS.get("outputs", PATHS.get("root", base_dir) / "outputs")
artifacts_dir = PATHS.get("artifacts", PATHS.get("root", base_dir) / "artifacts")

# Ensure required output directories exist
outputs_dir.mkdir(parents=True, exist_ok=True)
(outputs_dir / "predictions").mkdir(parents=True, exist_ok=True)
(outputs_dir / "tables").mkdir(parents=True, exist_ok=True)
(outputs_dir / "figures").mkdir(parents=True, exist_ok=True)
artifacts_dir.mkdir(parents=True, exist_ok=True)

## 1. Validation Set Evaluation

Validation predictions generated by Notebook 02 (`Logistic Regression`) and Notebook 03 (`Neural Network`) are loaded to compute baseline metrics on the validation split (`02-lr_val.parquet` and `03-nn_val.parquet`).

In [2]:
# Load validation prediction artifacts
df_lr_val = pd.read_parquet(outputs_dir / "predictions" / "02-lr_val.parquet")
df_nn_val = pd.read_parquet(outputs_dir / "predictions" / "03-nn_val.parquet")

# Calculate validation metrics
metrics_lr_val = compute_metrics(
    df_lr_val["y_true"], df_lr_val["y_pred"], df_lr_val["y_proba_pos"]
)
metrics_nn_val = compute_metrics(
    df_nn_val["y_true"], df_nn_val["y_pred"], df_nn_val["y_proba_pos"]
)

val_metrics_df = pd.DataFrame(
    [
        {"model": "Logistic Regression (Val)", **metrics_lr_val},
        {"model": "Neural Network (Val)", **metrics_nn_val},
    ]
)

print("Validation Set Metrics Summary:")
print(val_metrics_df.to_string(index=False))

Validation Set Metrics Summary:
                    model  accuracy  precision  recall       f1  roc_auc           confusion_matrix
Logistic Regression (Val)    0.8914   0.885085  0.8996 0.892283 0.958051 [[2208, 292], [251, 2249]]
     Neural Network (Val)    0.8964   0.889544  0.9052 0.897304 0.959838 [[2219, 281], [237, 2263]]


## 2. Final Test Set Evaluation

The frozen `test` feature matrix is loaded once using `load_features("test", allow_test=True)`. Model artifacts (`artifacts/logreg.joblib` and `artifacts/nn_model.keras`) run inference on the test split to generate final predictions.

In [3]:
# Load test set features (allow_test=True satisfies the single-evaluation pipeline rule)
X_test, y_test, ids_test = load_features("test", allow_test=True)

# Load frozen model artifacts
logreg_model = joblib.load(artifacts_dir / "logreg.joblib")
nn_model = load_model(artifacts_dir / "nn_model.keras")

# Generate probabilities and binary predictions for Logistic Regression
lr_test_proba = logreg_model.predict_proba(X_test)[:, 1]
lr_test_pred = (lr_test_proba >= 0.5).astype(int)

# Generate probabilities and binary predictions for Neural Network
nn_test_proba = nn_model.predict(X_test, verbose=0).ravel()
nn_test_pred = (nn_test_proba >= 0.5).astype(int)

# Construct standardized test predictions DataFrame (Long Format)
df_lr_test = pd.DataFrame(
    {
        "id": ids_test,
        "y_true": y_test,
        "y_pred": lr_test_pred,
        "y_proba_pos": lr_test_proba,
        "model": "logistic_regression",
    }
)

df_nn_test = pd.DataFrame(
    {
        "id": ids_test,
        "y_true": y_test,
        "y_pred": nn_test_pred,
        "y_proba_pos": nn_test_proba,
        "model": "neural_network",
    }
)

df_test_predictions = pd.concat([df_lr_test, df_nn_test], ignore_index=True)

## 3. Metrics Comparison & Summary Table

Metrics (Accuracy, Precision, Recall, F1 Score, ROC-AUC) are calculated for both models across the test set. Results are saved to `outputs/tables/04-eval_metrics_comparison.csv`.

In [4]:
# Calculate test set metrics
metrics_lr_test = compute_metrics(y_test, lr_test_pred, lr_test_proba)
metrics_nn_test = compute_metrics(y_test, nn_test_pred, nn_test_proba)

# Combine validation and test metrics into summary DataFrame
eval_summary_df = pd.DataFrame(
    [
        {"split": "val", "model": "logistic_regression", **metrics_lr_val},
        {"split": "val", "model": "neural_network", **metrics_nn_val},
        {"split": "test", "model": "logistic_regression", **metrics_lr_test},
        {"split": "test", "model": "neural_network", **metrics_nn_test},
    ]
)

# Export summary table to CSV
output_table_path = outputs_dir / "tables" / "04-eval_metrics_comparison.csv"
eval_summary_df.to_csv(output_table_path, index=False)
print(f"Metrics table written to: {output_table_path}")

Metrics table written to: /Users/yesidcardenas/movie-review-sentiment/outputs/tables/04-eval_metrics_comparison.csv


## 4. Visualizations

Three figures are generated for model evaluation:
1. **ROC Curves:** `04-eval_roc_comparison.png`
2. **Confusion Matrices:** `04-eval_confusion_matrices.png`
3. **Metrics Comparison Bar Plot:** `04-eval_comparison_metrics.png`

In [5]:
fig_dir = outputs_dir / "figures"

# 1. ROC Curve Comparison Plot
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_test_proba)
fpr_nn, tpr_nn, _ = roc_curve(y_test, nn_test_proba)

auc_lr = roc_auc_score(y_test, lr_test_proba)
auc_nn = roc_auc_score(y_test, nn_test_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr_lr, tpr_lr, label=f"Logistic Regression (AUC = {auc_lr:.4f})", linewidth=2)
plt.plot(fpr_nn, tpr_nn, label=f"Neural Network (AUC = {auc_nn:.4f})", linewidth=2, linestyle="--")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic (ROC) - Test Split")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(fig_dir / "04-eval_roc_comparison.png", dpi=300)
plt.close()

# 2. Side-by-Side Confusion Matrices
cm_lr = confusion_matrix(y_test, lr_test_pred)
cm_nn = confusion_matrix(y_test, nn_test_pred)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(
    cm_lr,
    annot=True,
    fmt="d",
    cmap="Blues",
    ax=axes[0],
    cbar=False,
    xticklabels=["Legitimate", "Phishing"],
    yticklabels=["Legitimate", "Phishing"],
)
axes[0].set_title("Logistic Regression - Test Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

sns.heatmap(
    cm_nn,
    annot=True,
    fmt="d",
    cmap="Greens",
    ax=axes[1],
    cbar=False,
    xticklabels=["Legitimate", "Phishing"],
    yticklabels=["Legitimate", "Phishing"],
)
axes[1].set_title("Neural Network - Test Confusion Matrix")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")

plt.tight_layout()
plt.savefig(fig_dir / "04-eval_confusion_matrices.png", dpi=300)
plt.close()

# 3. Bar Chart Comparing Test Set Metrics
test_metrics_melted = eval_summary_df[eval_summary_df["split"] == "test"].melt(
    id_vars=["model", "split"],
    value_vars=["accuracy", "precision", "recall", "f1", "roc_auc"],
    var_name="metric",
    value_name="score",
)

plt.figure(figsize=(10, 6))
chart = sns.barplot(
    data=test_metrics_melted,
    x="metric",
    y="score",
    hue="model",
    palette="Blues_d",
)
plt.title("Model Performance Comparison on Test Set")
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.xlabel("Metric")

for p in chart.patches:
    if p.get_height() > 0:
        chart.annotate(
            f"{p.get_height():.3f}",
            (p.get_x() + p.get_width() / 2.0, p.get_height()),
            ha="center",
            va="center",
            xytext=(0, 5),
            textcoords="offset points",
            fontsize=9,
        )

plt.legend(title="Model", loc="lower right")
plt.tight_layout()
plt.savefig(fig_dir / "04-eval_comparison_metrics.png", dpi=300)
plt.close()

print("All evaluation figures successfully saved to outputs/figures/")

All evaluation figures successfully saved to outputs/figures/


## 5. Write Handoff Artifacts

The final combined test prediction dataset is exported in long format to `outputs/predictions/04-test_predictions.parquet` for consumption by Notebook 05 (`05_divergence_judge.ipynb`).

In [6]:
pred_output_path = outputs_dir / "predictions" / "04-test_predictions.parquet"
df_test_predictions.to_parquet(pred_output_path, index=False)

print(f"Handoff artifact saved")

Handoff artifact saved
